In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

DATA = Path("data")
con = duckdb.connect()
con.execute("SET memory_limit = '8GB'")

for name in ["articles", "customers", "transactions_train"]:
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM '{(DATA / f'{name}.parquet').as_posix()}'")

LAST_DAY = "2020-09-22"
SAMPLE_FRAC = 0.20
SEED = 42
WEEKS_KEPT = 26

print(con.execute("SHOW TABLES").df())

                 name
0            articles
1           customers
2  transactions_train


In [2]:
cust = con.execute("SELECT customer_id FROM customers ORDER BY customer_id").df()

rng = np.random.default_rng(SEED)
sample = cust[rng.random(len(cust)) < SAMPLE_FRAC].reset_index(drop=True)

con.register("sample_df", sample)
con.execute(f"COPY sample_df TO '{(DATA / 'sample_customers.parquet').as_posix()}' (FORMAT parquet)")

print(f"All customers:     {len(cust):,}")
print(f"Sampled customers: {len(sample):,}  ({len(sample)/len(cust):.2%})")

All customers:     1,371,980
Sampled customers: 274,259  (19.99%)


In [3]:
con.execute(f"CREATE OR REPLACE VIEW sample_customers AS SELECT * FROM '{(DATA / 'sample_customers.parquet').as_posix()}'")

compare = con.execute("""
    WITH buys AS (
        SELECT customer_id, COUNT(DISTINCT t_dat) AS visit_days
        FROM transactions_train
        GROUP BY customer_id
    ),
    base AS (
        SELECT c.customer_id, c.age,
               COALESCE(b.visit_days, 0) AS visit_days,
               s.customer_id IS NOT NULL AS in_sample
        FROM customers c
        LEFT JOIN buys b USING (customer_id)
        LEFT JOIN sample_customers s USING (customer_id)
    )
    SELECT 'all' AS grp, COUNT(*) AS customers,
           MEDIAN(age) AS median_age,
           ROUND(AVG(visit_days), 2) AS avg_visit_days,
           ROUND(100.0 * AVG((visit_days = 0)::INT), 2) AS pct_never_bought,
           ROUND(100.0 * AVG((visit_days = 1)::INT), 2) AS pct_one_day
    FROM base
    UNION ALL
    SELECT 'sample', COUNT(*), MEDIAN(age),
           ROUND(AVG(visit_days), 2),
           ROUND(100.0 * AVG((visit_days = 0)::INT), 2),
           ROUND(100.0 * AVG((visit_days = 1)::INT), 2)
    FROM base WHERE in_sample
""").df()
print(compare.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   grp  customers  median_age  avg_visit_days  pct_never_bought  pct_one_day
   all    1371980        32.0            6.62              0.71        32.65
sample     274259        32.0            6.63              0.70        32.59


In [4]:
out = (DATA / "sample_transactions.parquet").as_posix()

con.execute(f"""
    COPY (
        SELECT t.t_dat,
               t.customer_id,
               t.article_id,
               t.price,
               t.sales_channel_id,
               DATE_DIFF('day', t.t_dat, DATE '{LAST_DAY}') // 7 AS w
        FROM transactions_train t
        JOIN sample_customers s USING (customer_id)
        WHERE DATE_DIFF('day', t.t_dat, DATE '{LAST_DAY}') // 7 < {WEEKS_KEPT}
        ORDER BY t.t_dat
    ) TO '{out}' (FORMAT parquet, COMPRESSION zstd)
""")

print(con.execute(f"""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT customer_id) AS customers,
           COUNT(DISTINCT article_id) AS articles,
           MIN(t_dat) AS first_day, MAX(t_dat) AS last_day,
           MIN(w) AS min_w, MAX(w) AS max_w
    FROM '{out}'
""").df().to_string(index=False))
print(f"File size: {Path(out).stat().st_size / 1024**2:.1f} MB")

   rows  customers  articles  first_day   last_day  min_w  max_w
1618007     148628     39709 2020-03-25 2020-09-22      0     25
File size: 20.6 MB


## Sampling: what I did and why

I'm on a laptop with 16 GB of RAM and no GPU, so I didn't model all 1.37 million customers. I took a random 20% of the customer table (274,259 people) with a fixed seed, so I get the same sample every time I run this. I sampled from the customer table rather than from purchases, so people who never bought anything can still show up, like they would in real life.

To check I hadn't been unlucky, I compared the sample with everyone. Median age, average shopping days, and the share of one-day and never-buying customers all matched to within 0.1 percentage points.

I also kept only the last 26 weeks of purchases. The EDA showed recent weeks matter much more than old ones in fashion. This leaves 1.6 million rows, which fits easily in memory.

The cost: only 54% of my sampled customers bought anything in those 26 weeks. If one of the others buys in the test week, my models will treat them like a new customer even though they shopped with H&M before. If that hurts results, I'll bring back older history for the "bought before" method only.